# 🫁 Asthma Disease Prediction System
**DSBD (Data Science & Big Data) mini project - T.Y. B.Tech CSD**

Is notebook ko Google Colab me upar se neeche **Runtime > Run all** karke chala sakte ho.

## 1. Problem statement, objectives, scope

**Problem statement:** Asthma ek chronic respiratory disease hai jiska early detection mushkil hota hai kyunki symptoms
(wheezing, cough, breathlessness) dusri bimariyon se milte-julte hain. Patient ki demographic, lifestyle,
environmental, medical-history, lung-function aur symptom data se yeh predict karna ki patient ko asthma hai ya nahi.

**Objectives**
1. Asthma dataset ko **PySpark** (Big Data layer) se load, clean aur feature-engineer karna.
2. EDA se asthma ke risk factors samajhna.
3. Class imbalance ko **SMOTE** se handle karna.
4. 4 ML models (Logistic Regression, Decision Tree, Random Forest, XGBoost) train + tune karna.
5. Accuracy, Precision, Recall, F1, ROC-AUC se compare karke best model chunna.
6. Best model ko save karke **Streamlit** web app me deploy karna.

**Scope:** Binary classification (Asthma: Yes/No) on tabular data. Yeh ek educational project hai,
medical diagnosis tool nahi.

**Big Data angle:** Dataset chhota hai (~2400 rows), lekin pipeline PySpark me likhi hai (Spark DataFrame, Imputer,
VectorAssembler, StandardScaler, MLlib models, CrossValidator) jo same code lakhs/crores rows par cluster par bhi chalta hai.

## 2. Dataset
**Kaggle:** `rabieelkharoubi/asthma-disease-dataset` ("Asthma Disease Dataset"), ek synthetic dataset, ~2,400 patients.
Agar Kaggle download fail ho to Kaggle se CSV manually download karke upload kar dena (code me fallback hai).

| Group | Columns |
|---|---|
| ID / useless | `PatientID`, `DoctorInCharge` (drop karenge) |
| Demographics | `Age`, `Gender` (0 male, 1 female), `Ethnicity` (0-3), `EducationLevel` (0-3), `BMI` |
| Lifestyle | `Smoking`, `PhysicalActivity`, `DietQuality`, `SleepQuality` |
| Environment | `PollutionExposure`, `PollenExposure`, `DustExposure` |
| Medical history | `PetAllergy`, `FamilyHistoryAsthma`, `HistoryOfAllergies`, `Eczema`, `HayFever`, `GastroesophagealReflux` |
| Lung function | `LungFunctionFEV1`, `LungFunctionFVC` (peak-flow ki jagah yeh lung-function measures hain) |
| Symptoms | `Wheezing`, `ShortnessOfBreath`, `ChestTightness`, `Coughing`, `NighttimeSymptoms`, `ExerciseInduced` |
| **Target** | `Diagnosis` (1 = asthma, 0 = no asthma) |

> Column names Kaggle page se match kar lena. Agar koi naam alag ho to sirf neeche ki column lists me badalna.

In [ ]:
# [COLAB-ONLY] Libraries install (Colab me ek baar)
!pip -q install pyspark xgboost imbalanced-learn kagglehub streamlit

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Project folders (GitHub structure ke hisaab se)
for folder in ["data", "models", "app", "images"]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
# [LOAD-DATA] Dataset download (Kaggle) ya manual upload
CSV_PATH = None
try:
    import kagglehub
    folder = kagglehub.dataset_download("rabieelkharoubi/asthma-disease-dataset")
    CSV_PATH = glob.glob(os.path.join(folder, "*.csv"))[0]
except Exception as e:
    print("Kaggle download nahi hua:", e)
    print("Kaggle se CSV download karke yahan upload karo:")
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

print("Using file:", CSV_PATH)

## 3. Big Data layer (PySpark)
Steps: Spark session > load > null check > clean > outliers > feature engineering > VectorAssembler > Spark MLlib.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("AsthmaPrediction")
         .master("local[*]")
         .config("spark.driver.memory", "2g")
         .getOrCreate())

sdf = spark.read.csv(CSV_PATH, header=True, inferSchema=True)
print("Rows:", sdf.count(), "| Columns:", len(sdf.columns))
sdf.printSchema()

In [ ]:
# Pehli 5 rows aur null counts
print(sdf.limit(5).toPandas().T)

null_counts = sdf.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf.columns]).toPandas().T
null_counts.columns = ["null_count"]
print("\nNull values per column:")
print(null_counts[null_counts["null_count"] > 0] if (null_counts["null_count"] > 0).any() else "No nulls found")

In [ ]:
# ---------- CLEANING ----------
from pyspark.ml.feature import Imputer

# 1) ID / useless columns hatao
sdf = sdf.drop("PatientID", "DoctorInCharge")

# 2) Duplicate rows hatao
before = sdf.count()
sdf = sdf.dropDuplicates()
print("Duplicates removed:", before - sdf.count())

# 3) Sab columns ko double me convert (Imputer ko numeric chahiye)
for c, t in sdf.dtypes:
    sdf = sdf.withColumn(c, F.col(c).cast("double"))

# 4) Target null ho to row hatao
sdf = sdf.dropna(subset=["Diagnosis"])

# 5) Baaki columns ke nulls ko median se fill karo
feature_cols_raw = [c for c in sdf.columns if c != "Diagnosis"]
imputed_names = [c + "_imp" for c in feature_cols_raw]
imputer = Imputer(inputCols=feature_cols_raw, outputCols=imputed_names, strategy="median")
sdf = imputer.fit(sdf).transform(sdf)
for c in feature_cols_raw:
    sdf = sdf.drop(c).withColumnRenamed(c + "_imp", c)

print("Clean rows:", sdf.count())

In [ ]:
# ---------- OUTLIERS (IQR capping) ----------
# Extreme values ko hatane ki jagah lower/upper limit par "cap" karte hain
continuous_for_outliers = ["Age", "BMI", "LungFunctionFEV1", "LungFunctionFVC"]

for c in continuous_for_outliers:
    q1, q3 = sdf.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = sdf.filter((F.col(c) < low) | (F.col(c) > high)).count()
    sdf = sdf.withColumn(
        c, F.when(F.col(c) < low, low).when(F.col(c) > high, high).otherwise(F.col(c)))
    print(f"{c}: bounds=({low:.2f}, {high:.2f}) outliers capped={n_out}")

In [ ]:
# ---------- FEATURE ENGINEERING ----------
symptom_cols = ["Wheezing", "ShortnessOfBreath", "ChestTightness", "Coughing", "NighttimeSymptoms", "ExerciseInduced"]
allergy_cols = ["PetAllergy", "HistoryOfAllergies", "Eczema", "HayFever"]

sdf = (sdf
       # FEV1/FVC ratio: obstructive lung disease (asthma) me ratio kam hota hai
       .withColumn("FEV1_FVC_Ratio",
                   F.when(F.col("LungFunctionFVC") > 0, F.col("LungFunctionFEV1") / F.col("LungFunctionFVC")).otherwise(F.lit(0.0)))
       # Kitne symptoms ek saath hain (0-6)
       .withColumn("SymptomScore", sum(F.col(c) for c in symptom_cols))
       # Kitni allergy-related conditions hain (0-4)
       .withColumn("AllergyScore", sum(F.col(c) for c in allergy_cols)))

sdf.select("FEV1_FVC_Ratio", "SymptomScore", "AllergyScore", "Diagnosis").limit(5).toPandas()

In [ ]:
# ---------- VectorAssembler + StandardScaler + Spark MLlib ----------
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (LogisticRegression as SparkLR,
                                       DecisionTreeClassifier as SparkDT,
                                       RandomForestClassifier as SparkRF)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

spark_feature_cols = [c for c in sdf.columns if c != "Diagnosis"]

# VectorAssembler: saare feature columns ko ek vector column me jodta hai (MLlib ki requirement)
assembler = VectorAssembler(inputCols=spark_feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

prep_model = Pipeline(stages=[assembler, scaler]).fit(sdf)
ml_df = prep_model.transform(sdf).select("features", F.col("Diagnosis").alias("label"))

train_sdf, test_sdf = ml_df.randomSplit([0.8, 0.2], seed=RANDOM_STATE)
print("Spark train rows:", train_sdf.count(), "| test rows:", test_sdf.count())

auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

spark_models = {
    "Spark Logistic Regression": SparkLR(featuresCol="features", labelCol="label", maxIter=50),
    "Spark Decision Tree": SparkDT(featuresCol="features", labelCol="label", maxDepth=5, seed=RANDOM_STATE),
    "Spark Random Forest": SparkRF(featuresCol="features", labelCol="label", numTrees=100, seed=RANDOM_STATE),
}

spark_rows = []
for name, est in spark_models.items():
    pred = est.fit(train_sdf).transform(test_sdf)
    spark_rows.append({"Model": name, "Accuracy": acc_eval.evaluate(pred),
                       "F1": f1_eval.evaluate(pred), "ROC_AUC": auc_eval.evaluate(pred)})

# Hyperparameter tuning (Spark CrossValidator) - Random Forest
rf = SparkRF(featuresCol="features", labelCol="label", seed=RANDOM_STATE)
grid = (ParamGridBuilder()
        .addGrid(rf.numTrees, [50, 100])
        .addGrid(rf.maxDepth, [5, 10])
        .build())
cv = CrossValidator(estimator=rf, estimatorParamMaps=grid, evaluator=auc_eval, numFolds=3, seed=RANDOM_STATE)
pred = cv.fit(train_sdf).transform(test_sdf)
spark_rows.append({"Model": "Spark Random Forest (CV tuned)", "Accuracy": acc_eval.evaluate(pred),
                   "F1": f1_eval.evaluate(pred), "ROC_AUC": auc_eval.evaluate(pred)})

spark_results = pd.DataFrame(spark_rows).round(4)
print("Spark MLlib results:")
spark_results

In [ ]:
# Clean data ko pandas me laao (data chhota hai) aur CSV save karo: sklearn + EDA + GitHub ke liye
pdf = sdf.toPandas()
pdf["Diagnosis"] = pdf["Diagnosis"].astype(int)
pdf.to_csv("data/asthma_clean.csv", index=False)
print(pdf.shape)
pdf.head()

## 4. EDA (Exploratory Data Analysis)

In [ ]:
# 4.1 Class balance
counts = pdf["Diagnosis"].value_counts().sort_index()
print(counts, "\n", (counts / len(pdf) * 100).round(1).astype(str) + " %")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x="Diagnosis", data=pdf, palette=["#4C9F70", "#D1495B"], ax=ax)
ax.set_xticklabels(["No asthma (0)", "Asthma (1)"])
ax.set_title("Class balance")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.savefig("images/class_balance.png", dpi=150); plt.show()

In [ ]:
# 4.2 Distributions (Asthma vs No asthma)
cols = ["Age", "BMI", "LungFunctionFEV1", "FEV1_FVC_Ratio"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, c in zip(axes.ravel(), cols):
    sns.histplot(data=pdf, x=c, hue="Diagnosis", kde=True, bins=30, palette=["#4C9F70", "#D1495B"], ax=ax)
    ax.set_title(f"{c} distribution")
plt.tight_layout(); plt.savefig("images/distributions.png", dpi=150); plt.show()

In [ ]:
# 4.3 Symptom prevalence: kitne % patients me symptom hai
symptoms = ["Wheezing", "ShortnessOfBreath", "ChestTightness", "Coughing", "NighttimeSymptoms", "ExerciseInduced"]
sym = pdf.groupby("Diagnosis")[symptoms].mean().T * 100
sym.columns = ["No asthma", "Asthma"]
sym.plot(kind="bar", figsize=(9, 4), color=["#4C9F70", "#D1495B"])
plt.ylabel("% patients with symptom"); plt.title("Symptom prevalence by diagnosis")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig("images/symptoms.png", dpi=150); plt.show()

In [ ]:
# 4.4 Correlation heatmap
corr = pdf.corr(numeric_only=True)
plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.3, cbar_kws={"shrink": 0.7})
plt.title("Correlation heatmap")
plt.tight_layout(); plt.savefig("images/correlation_heatmap.png", dpi=150); plt.show()

print("Diagnosis ke saath top correlations:")
print(corr["Diagnosis"].drop("Diagnosis").abs().sort_values(ascending=False).head(10).round(3))

## 5. Preprocessing (scikit-learn)
- **Encoding:** `Ethnicity`, `EducationLevel` = categorical codes, isliye **One-Hot Encoding**.
- **Scaling:** continuous columns par **StandardScaler**. Binary 0/1 columns jaise hain waise (passthrough).
- **Imbalance:** **SMOTE** (synthetic minority samples) sirf **training data** par, pipeline ke andar.
  Test data par kabhi SMOTE nahi lagate, warna results jhoothe aate hain (data leakage).
- **Split:** 80/20 stratified (dono sets me same class ratio).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from imblearn.over_sampling import SMOTE

CAT_COLS = ["Ethnicity", "EducationLevel"]
NUM_COLS = ["Age", "BMI", "PhysicalActivity", "DietQuality", "SleepQuality",
            "PollutionExposure", "PollenExposure", "DustExposure",
            "LungFunctionFEV1", "LungFunctionFVC", "FEV1_FVC_Ratio", "SymptomScore", "AllergyScore"]
BIN_COLS = ["Gender", "Smoking", "PetAllergy", "FamilyHistoryAsthma", "HistoryOfAllergies", "Eczema",
            "HayFever", "GastroesophagealReflux", "Wheezing", "ShortnessOfBreath", "ChestTightness",
            "Coughing", "NighttimeSymptoms", "ExerciseInduced"]
FEATURES = CAT_COLS + NUM_COLS + BIN_COLS

X = pdf[FEATURES]
y = pdf["Diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ("num", StandardScaler(), NUM_COLS),
    ("bin", "passthrough", BIN_COLS),
])

# Sirf demo ke liye: SMOTE se pehle aur baad ki class counts
X_demo = clone(preprocess).fit_transform(X_train)
X_res, y_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X_demo, y_train)
print("Train class counts BEFORE SMOTE:", y_train.value_counts().to_dict())
print("Train class counts AFTER  SMOTE:", pd.Series(y_res).value_counts().to_dict())
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 6. Models + hyperparameter tuning
Har model ek **pipeline** me: `preprocess > SMOTE > classifier`. **GridSearchCV** (3-fold, ROC-AUC) best parameters dhundta hai.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

configs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        {"clf__C": [0.1, 1, 10]}),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {"clf__max_depth": [3, 5, 8], "clf__min_samples_split": [2, 10]}),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [5, 10, None]}),
    "XGBoost": (
        XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1]}),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
trained = {}

for name, (clf, params) in configs.items():
    pipe = ImbPipeline([
        ("prep", preprocess),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", clf),
    ])
    gs = GridSearchCV(pipe, params, cv=cv, scoring="roc_auc", n_jobs=-1)
    gs.fit(X_train, y_train)
    trained[name] = gs.best_estimator_
    print(f"{name:20s} | best CV ROC-AUC = {gs.best_score_:.4f} | best params = {gs.best_params_}")

## 7. Evaluation

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)

rows, preds, probs = [], {}, {}
for name, model in trained.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    preds[name], probs[name] = y_pred, y_prob
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False).reset_index(drop=True).round(4)
results_df.to_csv("data/model_comparison.csv", index=False)
results_df

In [ ]:
# Comparison chart
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1.05))
plt.title("Model comparison (test set)")
plt.xticks(rotation=20, ha="right"); plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig("images/model_comparison.png", dpi=150); plt.show()

# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
for ax, name in zip(axes.ravel(), trained):
    cm = confusion_matrix(y_test, preds[name])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["No asthma", "Asthma"], yticklabels=["No asthma", "Asthma"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.savefig("images/confusion_matrices.png", dpi=150); plt.show()

# ROC curves
plt.figure(figsize=(6, 5))
for name in trained:
    fpr, tpr, _ = roc_curve(y_test, probs[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, probs[name]):.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC curves")
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig("images/roc_curves.png", dpi=150); plt.show()

In [ ]:
# Best model selection (highest ROC-AUC)
best_name = results_df.loc[0, "Model"]
best_model = trained[best_name]
print("Best model:", best_name)
print(classification_report(y_test, preds[best_name], target_names=["No asthma", "Asthma"]))

# Feature importance (tree models) / coefficients (logistic regression)
try:
    names = best_model.named_steps["prep"].get_feature_names_out()
    clf = best_model.named_steps["clf"]
    importance = clf.feature_importances_ if hasattr(clf, "feature_importances_") else np.abs(clf.coef_[0])
    top = pd.Series(importance, index=names).sort_values(ascending=False).head(15)
    top.sort_values().plot(kind="barh", figsize=(7, 5), color="#2E86AB")
    plt.title(f"Top 15 important features ({best_name})")
    plt.tight_layout(); plt.savefig("images/feature_importance.png", dpi=150); plt.show()
except Exception as e:
    print("Feature importance skipped:", e)

## 8. Model save (joblib) + Streamlit app

In [ ]:
import joblib

bundle = {
    "model": best_model,            # poori pipeline (preprocess + classifier)
    "model_name": best_name,
    "features": FEATURES,           # app ko column order chahiye
    "metrics": results_df.iloc[0].to_dict(),
}
joblib.dump(bundle, "models/asthma_model.joblib")
print("Saved: models/asthma_model.joblib")

# Quick check: saved model reload karke 1 patient par predict
loaded = joblib.load("models/asthma_model.joblib")
sample = X_test.iloc[[0]]
print("Predicted probability of asthma:", round(float(loaded["model"].predict_proba(sample)[0, 1]), 3),
      "| Actual:", int(y_test.iloc[0]))

In [ ]:
%%writefile app/app.py
"""Asthma Prediction System - Streamlit web app.

Run from the project root:
    streamlit run app/app.py
"""
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Asthma Prediction System", page_icon="🫁", layout="centered")

MODEL_PATH = Path(__file__).resolve().parent.parent / "models" / "asthma_model.joblib"


@st.cache_resource
def load_bundle():
    # bundle = {"model": pipeline, "model_name": str, "features": [...], "metrics": {...}}
    return joblib.load(MODEL_PATH)


def yes_no(label, key):
    # Streamlit checkbox -> 1 / 0
    return 1 if st.checkbox(label, key=key) else 0


bundle = load_bundle()
model = bundle["model"]
FEATURES = bundle["features"]

st.title("🫁 Asthma Disease Prediction System")
st.caption(f"Model: {bundle['model_name']}  |  DSBD mini project (educational use only)")

st.subheader("1. Patient details")
c1, c2 = st.columns(2)
age = c1.slider("Age (years)", 5, 80, 30)
gender = c2.selectbox("Gender", ["Male", "Female"])
ethnicity = c1.selectbox("Ethnicity", ["Caucasian", "African American", "Asian", "Other"])
education = c2.selectbox("Education level", ["None", "High School", "Bachelor's", "Higher"])
bmi = c1.slider("BMI", 15.0, 40.0, 24.0, 0.1)
smoking = c2.selectbox("Smoker?", ["No", "Yes"])

st.subheader("2. Lifestyle and environment (0 = low, 10 = high)")
c1, c2 = st.columns(2)
physical = c1.slider("Physical activity", 0.0, 10.0, 5.0, 0.1)
diet = c2.slider("Diet quality", 0.0, 10.0, 5.0, 0.1)
sleep = c1.slider("Sleep quality (4-10)", 4.0, 10.0, 7.0, 0.1)
pollution = c2.slider("Pollution exposure", 0.0, 10.0, 5.0, 0.1)
pollen = c1.slider("Pollen exposure", 0.0, 10.0, 5.0, 0.1)
dust = c2.slider("Dust exposure", 0.0, 10.0, 5.0, 0.1)

st.subheader("3. Medical history")
c1, c2 = st.columns(2)
with c1:
    family = yes_no("Family history of asthma", "family")
    allergies = yes_no("History of allergies", "allergies")
    pet = yes_no("Pet allergy", "pet")
with c2:
    eczema = yes_no("Eczema", "eczema")
    hay = yes_no("Hay fever", "hay")
    gerd = yes_no("Acid reflux (GERD)", "gerd")

st.subheader("4. Lung function")
c1, c2 = st.columns(2)
fev1 = c1.slider("FEV1 (litres)", 1.0, 4.0, 2.5, 0.01)
fvc = c2.slider("FVC (litres)", 1.5, 6.0, 3.5, 0.01)

st.subheader("5. Symptoms")
c1, c2 = st.columns(2)
with c1:
    wheezing = yes_no("Wheezing", "wheezing")
    breath = yes_no("Shortness of breath", "breath")
    chest = yes_no("Chest tightness", "chest")
with c2:
    cough = yes_no("Coughing", "cough")
    night = yes_no("Nighttime symptoms", "night")
    exercise = yes_no("Exercise-induced symptoms", "exercise")

if st.button("Predict", type="primary"):
    row = {
        "Age": age,
        "Gender": 0 if gender == "Male" else 1,
        "Ethnicity": ["Caucasian", "African American", "Asian", "Other"].index(ethnicity),
        "EducationLevel": ["None", "High School", "Bachelor's", "Higher"].index(education),
        "BMI": bmi,
        "Smoking": 1 if smoking == "Yes" else 0,
        "PhysicalActivity": physical,
        "DietQuality": diet,
        "SleepQuality": sleep,
        "PollutionExposure": pollution,
        "PollenExposure": pollen,
        "DustExposure": dust,
        "PetAllergy": pet,
        "FamilyHistoryAsthma": family,
        "HistoryOfAllergies": allergies,
        "Eczema": eczema,
        "HayFever": hay,
        "GastroesophagealReflux": gerd,
        "LungFunctionFEV1": fev1,
        "LungFunctionFVC": fvc,
        "Wheezing": wheezing,
        "ShortnessOfBreath": breath,
        "ChestTightness": chest,
        "Coughing": cough,
        "NighttimeSymptoms": night,
        "ExerciseInduced": exercise,
    }
    # Same engineered features as the notebook
    row["FEV1_FVC_Ratio"] = fev1 / fvc
    row["SymptomScore"] = wheezing + breath + chest + cough + night + exercise
    row["AllergyScore"] = pet + allergies + eczema + hay

    X = pd.DataFrame([row])[FEATURES].astype(float)
    prob = float(model.predict_proba(X)[0, 1])

    st.divider()
    st.metric("Asthma probability", f"{prob * 100:.1f}%")
    st.progress(min(max(prob, 0.0), 1.0))
    if prob >= 0.5:
        st.error("Higher risk of asthma. Please consult a doctor for proper diagnosis.")
    else:
        st.success("Lower risk of asthma based on the entered values.")

st.info("This is a student project built on a public dataset. It is not a medical device and "
        "must not be used for real diagnosis.")

In [ ]:
%%writefile app/requirements.txt
streamlit
pandas
numpy
scikit-learn
imbalanced-learn
xgboost
joblib

In [ ]:
%%writefile requirements.txt
pyspark
pandas
numpy
matplotlib
seaborn
scikit-learn
imbalanced-learn
xgboost
joblib
streamlit
kagglehub

### Streamlit app Colab me test karna (optional)
Asli run apne laptop par karna better hai (`streamlit run app/app.py`). Colab me test karna ho to neeche wala cell chalao,
`npx` ke baad aane wale link par jao aur password me jo IP print hua woh daalo.

In [ ]:
# [COLAB-ONLY] Optional: Colab me app ka temporary public link
!pip -q install streamlit
!streamlit run app/app.py &>/content/streamlit_logs.txt &
!curl -s ipv4.icanhazip.com
!npx -y localtunnel --port 8501

## 9. Folder structure, GitHub aur README
```
asthma-prediction-system/
├── data/
│   ├── asthma_clean.csv          # cleaned data (Spark output)
│   └── model_comparison.csv
├── notebooks/
│   └── Asthma_Prediction_DSBD.ipynb
├── app/
│   ├── app.py                    # Streamlit app
│   └── requirements.txt          # light requirements (deployment)
├── models/
│   └── asthma_model.joblib
├── images/                       # EDA + result plots (README screenshots)
├── requirements.txt              # full requirements (notebook)
├── README.md
└── .gitignore
```
**README.md** (alag file me diya hai) ko repo ke root me rakho.

**Neeche wala cell** poora project ek ZIP me download kar dega. ZIP kholke sab kuch apne repo folder me daalo, phir GitHub steps follow karo:
```bash
git clone https://github.com/<username>/asthma-prediction-system.git
cd asthma-prediction-system
# ZIP ki files yahan copy karo
git add data/ notebooks/ && git commit -m "Add data and notebook"
git add models/ app/ && git commit -m "Add trained model and Streamlit app"
git add images/ README.md requirements.txt .gitignore && git commit -m "Add docs and plots"
git push origin main
```

In [ ]:
# [COLAB-ONLY] Notebook ko pehle File > Download .ipynb karke notebooks/ folder ke liye rakh lo.
# Yeh cell data, models, app, images ko ZIP karke download karta hai.
!zip -rq asthma_project_files.zip data models app images requirements.txt
from google.colab import files
files.download("asthma_project_files.zip")